## RAG Agent

**R**etrieval **A**ugmented **G**eneration (RAG) is a powerful technique that enables AI agents to access and leverage external knowledge sources beyond their training data. In this tutorial, we'll build a RAG agent that can answer questions about the JFK assassination files using OpenAI's Agents SDK and Pinecone vector database.

RAG is particularly useful when:
- You need up-to-date information beyond the model's training cutoff
- You have domain-specific documents or proprietary data
- You want to reduce hallucinations by grounding responses in factual sources
- You need to cite sources for transparency and verification

By the end of this tutorial, you'll have built an agent that can search through historical documents and provide accurate, sourced answers about the JFK files.


### Prerequisites


Before we begin, let's install the required packages:

```bash
!pip install -qU \
    openai-agents==0.0.13 \
    psycopg2-binary==2.9.9 \
    pgvector==0.2.3 \
    sqlalchemy==2.0.25 \
    datasets==3.6.0 \
    semantic-chunkers==0.1.1
```

We'll also need API keys for OpenAI. You can get:
- An OpenAI API key from the [OpenAI Platform](https://platform.openai.com/api-keys)
Additionally you will need to settup your PostGresSQL database:
- You can download the software required from the [PostGresSQL website](https://www.postgresql.org/)

In [77]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass(
    "Enter OPENAI_API_KEY: "
)

### Testing LLM Knowledge Limitations

Before implementing RAG, let's first demonstrate why it's needed. We'll create a basic agent and test its knowledge about specific topics to show the limitations of relying solely on the model's training data.

In [78]:
from agents import Agent

agent = Agent(
    name="Agent",
    model="gpt-4.1-mini"
)

We'll ask our agent `"where was Oswald in october 1959?"`:

In [79]:
from agents import Runner

query = "where was Oswald in october 1959?"

result = await Runner.run(
    starting_agent=agent,
    input=query,
)

print(result.final_output)

2025-06-13 11:24:28 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


In October 1959, Lee Harvey Oswald was in the Soviet Union. He had traveled there earlier that year, arriving in October 1959. Oswald lived in Minsk and worked at a radio and television factory.


Oswald was _also_ in Helsinki, Finland in October 1959 according to the [JFK files](https://www.archives.gov/files/research/jfk/releases/2025/0318/104-10004-10156.pdf) - which our agent missed. We can try and tease out this information:

In [80]:
result = await Runner.run(
    starting_agent=agent,
    input=[
        {"role": "user", "content": query},
        {"role": "assistant", "content": result.final_output},
        {"role": "user", "content": "did he go anywhere else?"}
    ],
)

print(result.final_output)

2025-06-13 11:24:30 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Yes, after initially arriving in the Soviet Union in October 1959, Lee Harvey Oswald spent time in different locations within the USSR. He first stayed in Minsk, where he worked at the Radio and Television Factory. Later, in mid-1960, he moved to Moscow and applied for Soviet citizenship, though the application was denied. He eventually returned to the United States in June 1962. 

So, during the period starting October 1959, he was primarily in Minsk, then Moscow before returning to the U.S.


Our agent is clearly not aware of Oswald's trip to Helsinki - that is because the underlying LLM has not seen that information during it's training process. We call information learned during LLM training **parametric knowledge**, ie knowledge stored within the model _parameters_.

LLMs can also make use of **source knowledge** to answer questions. Source knowledge refers to information provided to an LLM via a prompt, either provided via the user, the LLM instructions, or in our case - via an external database - ie with **R**etrieval **A**ugmented **G**eneration (RAG). Before we build out our RAG pipeline, let's see if our LLM can answer our question when we provide the relevant information about Oswald's whereabouts via our `instructions`.

In [81]:
source_knowledge = (
    "~SECRET~\n"
    "1 June 1964\n"
    "\n"
    "## MEMO FOR THE RECORD\n"
    "\n"
    "1. At 0900 this morning I talked with Frank Friberg recently "
    "returned COS Helsinki re Warren Commission inquiry concerning "
    "the timetable of Oswald's stay in Finland in October 1959, including "
    "his contact with the Soviet Consulate there. (Copy of the Commission "
    "letter of 25 May 64 and State Cable of 22 May 64 attached.)"
)

agent = Agent(
    name="Agent",
    instructions=(
        "You are an assistant specialized in answering questions about the JFK assassination"
        "and related documents.\n"
        "Here is some additional context that you can use:\n"
        f"{source_knowledge}\n"
    ),
    model="gpt-4.1-mini"
)

Let's ask our original `query` again:

In [82]:
result = await Runner.run(
    starting_agent=agent,
    input=query,
)

print(result.final_output)

2025-06-13 11:24:32 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


According to the context provided in the memo dated 1 June 1964, Lee Harvey Oswald was in Finland in October 1959. The memo discusses an inquiry into the timetable of Oswald's stay in Finland during that time, including his contact with the Soviet Consulate in Helsinki.


Perfect, this is much better! Now what we just did works for this simple example, but it doesn't scale. If we want an agent that can answer any question and use context from _all_ of the JFK files, we need to build a RAG pipeline.

## Building a RAG Pipeline

A RAG pipeline actually requires two _core_ pipelines - an **ingestion pipeline** and a **retrieval pipeline**. At a high level those pipelines are responsible for:

* **Ingestion** handles the initial data preparation, embedding, and indexing. We'll explain those steps in more detail soon, but the tldr is that the ingestion pipeline will transform a set of unstructured and messy PDFs into a "second brain" for our agent, ie the _source knowledge_.

* **Retrieval** handles the query-time retrieval of information. It defines how we access and retrieve source knowledge from our second brain.

Naturally, we need to first develop our **ingestion pipeline** so that we can populate our second brain before we use the **retrieval pipeline** to retrieve anything.

### Ingestion Pipeline

The ingestion pipeline consists of three (or four) steps:

0. **Process the PDF** into plain text - with the `aurelio-ai/jfk-files` dataset (below) this step has been completed.

1. **Chunk** the plain text into smaller segments (a good rule of thumb is ~300-400 tokens per chunk).

2. **Embed** each chunk with OpenAI's `text-embedding-3-small` to create _vectors_.

3. **Index** those vectors in Pinecone with metadata like _source URL_, _document title_, etc.

![JFK document ingestion pipeline, covering PDF text to chunked text, embedding those chunks into semantically meaningful vector embeddings, and sending those vector embeddings to a vector database](../assets/jfk-ingestion-pipeline.png)

To begin, we'll start at step **0** and download the pre-parsed JFK files.


### Loading the JFK Files Dataset

We'll use a dataset of the JFK files, which we will pull from the [Hugging Face Hub](). This dataset contains historical documents that our agent will search through to answer questions:

In [83]:
from datasets import load_dataset

dataset = load_dataset(
    "aurelio-ai/jfk-files",
    split="train"
)

Let's examine a sample document to understand the data structure:

In [84]:
dataset[0]

{'id': 'doc_21c0d725_0fa9_40ef_a217_c062909cc236',
 'filename': '104-10110-10340.pdf',
 'url': 'https://www.archives.gov/files/research/jfk/releases/2025/0318/104-10110-10340.pdf',
 'date': datetime.datetime(2025, 3, 18, 0, 0),
 'content': '[704-10710-10340}\n\n<!-- image -->',
 'pages': 1}

Each document contains:
- id
- embedding
- text
- source
- title
- chunk_id

We'll need the text content from our chunks which we access via the `content` attribute:

### Embedding and Indexing


To enable semantic search over our documents, we'll use PostgreSQL with the pgvector extension - a powerful vector similarity search solution that integrates directly with PostgreSQL. Vector databases allow us to store and search through vector embeddings (numerical representations of text) to find semantically similar content. While there are many vector DB options out there, pgvector provides a robust, production-ready solution that leverages PostgreSQL's reliability and scalability.

First, let's set up our PostgreSQL connection details which we'll need to configure in our environment:

## Postgres Setup

Set the following environment variables before running this notebook:
- POSTGRES_HOST
- POSTGRES_PORT
- POSTGRES_DB
- POSTGRES_USER
- POSTGRES_PASSWORD

This notebook uses SQLAlchemy and pgvector for vector search.

In [89]:
import os

# Set Postgres environment variables
os.environ["POSTGRES_HOST"] = "localhost"  # your host
os.environ["POSTGRES_PORT"] = "5432"       # your port
os.environ["POSTGRES_DB"] = "jfk_db" # your database name
os.environ["POSTGRES_USER"] = "postgres"  # your username
os.environ["POSTGRES_PASSWORD"] = "your_password"  # your password

Next we want to establish a connection to make sure our PostGresSQL server is running as intended, we do this using the `psycopg2` library.

In [90]:
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT"),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)
print("Connected!")
conn.close()

Connected!


Now we want to create the database inside our server, as mentioned before we can follow our data variables (id, embedding, text, etc...) to create unique columns in our database, we also need to make a connection and create an extension for vectors assuming you haven't already done this.

In [91]:
import os
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker, declarative_base  # Updated import
from sqlalchemy import Column, String, Integer
from pgvector.sqlalchemy import Vector

Base = declarative_base()  # Using the new import

class JFKDocument(Base):
    __tablename__ = 'jfk_documents'
    id = Column(String, primary_key=True)
    embedding = Column(Vector(1536))
    text = Column(String)
    source = Column(String)
    title = Column(String)
    chunk_id = Column(Integer)

# Build the connection string
pg_uri = f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
engine = create_engine(pg_uri)
Session = sessionmaker(bind=engine)
session = Session()

# First, let's check if the extension is installed
with engine.connect() as conn:
    try:
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
        conn.commit()
    except Exception as e:
        print(f"Error creating extension: {e}")
        raise

# Create table if not exists
Base.metadata.create_all(engine)

To embed and index a chunk, we can do the following:

First create a client connection to OpenAI to use their embedding models.

In [92]:
from openai import OpenAI

client = OpenAI()

Next we can provide some basic texts we want to chunk as so.

In [93]:
texts = [
    'this is the first chunk of text',
    'then this is the second chunk of text'
]

Then using our client we can create embeddings, providing the model to use and our text inputs.

In [94]:
res = client.embeddings.create(
    input=texts,
    model="text-embedding-3-small"
)

2025-06-13 11:24:35 - httpx - INFO - _client.py:1025 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Then finally we can inspect our embeddings using our result previously, showcasing the amount of embeddings placed.

In [95]:
len(res.data), len(res.data[0].embedding)

(2, 1536)

Perfect! Now we simply repeat that process for all of our docs. We will do this in batches to avoid excessive network calls with small packages.

In [96]:
from tqdm.auto import tqdm
import tiktoken

def chunk_text(text, chunk_size=4000):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + chunk_size]
        chunks.append(tokenizer.decode(chunk))
    return chunks

def truncate_text(text, max_length=1000):
    if len(text) > max_length:
        return text[:max_length] + "..."
    return text

data = dataset.to_pandas()
batch_size = 100
for i in tqdm(range(0, len(data), batch_size)):
    i_end = min(len(data), i+batch_size)
    batch = data.iloc[i:i_end]
    all_ids = []
    all_texts = []
    all_metadata = []
    for idx, row in batch.iterrows():
        chunks = chunk_text(row['content'])
        for chunk_idx, chunk in enumerate(chunks):
            all_ids.append(f"{row['id']}-{chunk_idx}")
            all_texts.append(chunk)
            all_metadata.append({
                'text': truncate_text(chunk),
                'source': row['url'],
                'title': row['filename'],
                'chunk_id': chunk_idx
            })
    embeds = client.embeddings.create(
        input=all_texts,
        model="text-embedding-3-small"
    )
    vectors = [record.embedding for record in embeds.data]
    # Upsert all chunks into Postgres
    for doc_id, vector, meta in zip(all_ids, vectors, all_metadata):
        doc = JFKDocument(
            id=doc_id,
            embedding=vector,
            text=meta['text'],
            source=meta['source'],
            title=meta['title'],
            chunk_id=meta['chunk_id']
        )
        session.merge(doc)
    session.commit()

100%|██████████| 7/7 [00:15<00:00,  2.18s/it]


That's our **ingestion pipeline** complete and we're ready to move on to the **retrieval pipeline**.

## Retrieval Pipeline

Our retrieval pipeline is what will be used to retrieve the right source knowledge for our agent at query-time. We will be implementing this via an Agent SDK `@function_tool` but before we do so let's directly test retrieval.

As we're using PostgreSQL with pgvector, we'll need to:
1. Generate embeddings for our query using OpenAI's embedding model
2. Use pgvector's similarity search operator (`<=>`) to find the most semantically similar documents
3. Return the top matches to provide context for our agent

In [97]:
from agents import function_tool
from sqlalchemy import text
import numpy as np

@function_tool
def return_source_knowledge(query: str) -> str:
    """This tool gives you search access to the full JFK files. To use this tool you
    should provide search queries with as much context as possible, and using natural
    language to describe the query.

    This tool will return five of the most relevant document chunks for your query
    """
    try:
        # 1. Get the query embedding
        embeds_response = client.embeddings.create(
            input=[query],
            model="text-embedding-3-small"
        )
        query_embedding = np.array(embeds_response.data[0].embedding)

        # 2. Create a new connection for this query
        conn = psycopg2.connect(
            host=os.getenv("POSTGRES_HOST"),
            port=os.getenv("POSTGRES_PORT"),
            dbname=os.getenv("POSTGRES_DB"),
            user=os.getenv("POSTGRES_USER"),
            password=os.getenv("POSTGRES_PASSWORD")
        )
        
        try:
            with conn.cursor() as cur:
                # Cast the array to vector type
                cur.execute("""
                    SELECT text, source, title, chunk_id
                    FROM jfk_documents
                    ORDER BY embedding <=> %s::vector
                    LIMIT 3
                """, (query_embedding.tolist(),))
                rows = cur.fetchall()
                source_knowledge = "\n".join(row[0] for row in rows)
                return source_knowledge
        finally:
            conn.close()
    except Exception as e:
        print(f"Error in return_source_knowledge_2: {e}")
        return "Error retrieving source knowledge. Please try again."

Now we provide our `jfk_files_search` tool to an agent.

In [98]:
rag_agent = Agent(
    name="JFK Document Assistant",
    model="gpt-4o",
    instructions="""You are an assistant specialized in answering questions about the JFK assassination and related documents. 
    When users ask questions about JFK, the assassination, or related historical events, use the return_source_knowledge tool 
    to retrieve relevant information from the official JFK although not all files, only return what you have found from the tool.""",
    tools=[return_source_knowledge]
)

## Building the Final RAG Agent

Now we can use our agent to discover who really assassinated JFK. First, let's confirm our agent is functional with our original query about Oswald's whereabouts in October 1959.

In [99]:
query = "where was Lee Harvey Oswald in october 1959?"

result = await Runner.run(
    starting_agent=rag_agent, 
    input=query,
)

print(result.final_output)

2025-06-13 11:24:52 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-06-13 11:24:53 - httpx - INFO - _client.py:1025 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-06-13 11:24:55 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


In October 1959, Lee Harvey Oswald was in Helsinki, Finland. He was involved in a timetable of activities, including contact with the Soviet Consulate there.


In [100]:
from IPython.display import Markdown

query = "I see mentions of Oswald in Mexico, what did he do there?"

result = await Runner.run(
    starting_agent=rag_agent,
    input=query,
)

display(Markdown(result.final_output))

2025-06-13 11:27:00 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-06-13 11:27:01 - httpx - INFO - _client.py:1025 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-06-13 11:27:03 - httpx - INFO - _client.py:1740 - _send_single_request() - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


In September and October of 1963, Lee Harvey Oswald visited the Soviet Embassy in Mexico City. During this visit, he attempted to obtain a visa to return to the USSR. The consular officer handling his visa request was Valeriy Vladimirovich Kostikov. There’s no indication of any relationship between Oswald and Kostikov outside of this visa request.

This activity in Mexico was a part of Oswald’s broader movements and contacts in the lead-up to the JFK assassination.